# MPRALegNet for cell-selective enhancer design

[Notebook](multiple-models.ipynb) · [Installation](../docs/installation.md)

Favor K562 activity over HepG2 and WTC11. Each cell type uses its full ten-fold pretrained MPRALegNet ensemble. The objective combines the predictions on their native scale. Install MEME Suite with the notebook dependencies. The first model load downloads the pretrained checkpoint.

## 1. Load the model



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

import os
from pathlib import Path

import torch
from bioseq_models import load_model
from MotifSift import CombinedScorer, MultiTaskObjective, call_motifs, generate_designs, random_seed_sequences, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
cells = ["K562", "HepG2", "WTC11"]
model = CombinedScorer({
    cell: load_model(f"agarwal_{cell.lower()}", device=device)
    for cell in cells
})
objective = MultiTaskObjective(high=["K562"], low=["HepG2", "WTC11"])
seeds = random_seed_sequences(230, 100, seed=42)
run = Path("outputs/mpralegnet")
os.makedirs(run, exist_ok=True)




## 2. Generate candidates

Run the eleven optimizers and keep 100 candidates per optimizer. The saved tables show the previous full run. New runs use the current MotifSift defaults.



In [2]:
designs = generate_designs(
    model, length=230, initial=seeds, objective=objective,
    output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,2.606737,2.687437
basesearch,100,2.115711,2.250503
beam_search,100,2.562237,2.916238
cross_entropy_method,100,0.603610,0.793149
den,100,2.791457,2.861408
fast_seqprop,100,3.109218,3.340664
genetic_algorithm,100,3.142521,3.212666
ism_greedy,100,2.588911,2.883342
ledidi,100,1.738689,1.983505




## 3. Discover motifs

Extract positive seqlets from in silico mutagenesis scores and discover motifs with XSTREME.



In [3]:
motifs = call_motifs(
    designs, model=model,
    output_dir=run / "motifs",
)
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-WGATAAGR,1794,834,3460
1,2-CAGCTG,371,268,602
2,3-AATATTGC,451,279,228
4,5-TGATGCAATW,677,406,689
5,7-CTAACSGTYW,326,279,4165
6,8-CCAACG,393,325,1505
9,11-CTGGACAGC,411,283,699
11,14-AAGTTTCAGC,439,309,70
12,16-GTATTACAGT,85,85,55




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,GCGTCTGTTCTTAACTGATAAGATCTAGTACCCAACGGTTATAACG...,3.340664,1-WGATAAGR;2-CAGCTG;3-AATATTGC;5-TGATGCAATW;7-...
833,2,GAGTATAGATGTCCGTTAGACTCTAACGGGTTTGACACTCAGTTCC...,1.750315,7-CTAACSGTYW
619,3,GTGGGGTGAGGGGGGTATAGGGGTTGGCGCCCCGAGCCTTATCGCG...,2.549640,1-WGATAAGR;11-CTGGACAGC
571,4,TAACACCGGATAAACGCCTGCACCCTTAGATATTGGGGGGGACCCA...,2.577868,1-WGATAAGR;3-AATATTGC;5-TGATGCAATW
850,5,GCCTCGAAAAGGGCGATAAAACGGCTAGTCACAACTGTCCCAACCA...,1.734252,1-WGATAAGR;2-CAGCTG;7-CTAACSGTYW
815,6,GAGCATAGATGTCAGTTAGACTCTAACGGTTTTACCACTTAGTTCC...,1.780218,5-TGATGCAATW;7-CTAACSGTYW
826,7,GAGCATATATGTCCGTTAGACTCTAACGTTTTTGACACTCAGTTCC...,1.770923,1-WGATAAGR;7-CTAACSGTYW;8-CCAACG;16-GTATTACAGT
359,8,GGGGGGGGGTAGGGGTGGGGTGATGGCCCCCCTTGACACCCTCCCT...,2.682513,1-WGATAAGR;7-CTAACSGTYW;8-CCAACG;11-CTGGACAGC
700,9,AAGTGTTGTTATCTGATGCAACGCGCTGAAACTTACACCGACTAAA...,2.215343,1-WGATAAGR;2-CAGCTG;5-TGATGCAATW;7-CTAACSGTYW;...
853,10,GCCTCGATAAGGGCGATAAAACGGCTAGTTACAACTGTCCAAGCCA...,1.729610,1-WGATAAGR;2-CAGCTG;7-CTAACSGTYW;11-CTGGACAGC;...
